In [ ]:
# Cell 1: Cài đặt thư viện (nếu cần)
!pip install -q -U transformers accelerate peft bitsandbytes datasets trl qwen-vl-utils rouge-score nltk bert_score

In [ ]:
import os
import torch
import numpy as np
import shutil
from datasets import load_dataset, Image, DatasetDict, Dataset
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from transformers import EarlyStoppingCallback
from qwen_vl_utils import process_vision_info
from sklearn.metrics import accuracy_score
from tqdm import tqdm

import matplotlib.pyplot as plt
import json
import pandas as pd
from sklearn.model_selection import train_test_split

# Metrics
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import nltk
from collections import Counter
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bert_score

nltk.download('punkt', quiet=True)

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

In [ ]:
BASE_DATA = "/kaggle/input/vietnam-landmark-mqa-dataset"
BASE_INPUT = "/kaggle/input/vietnam-landmark-mqa-dataset"
NAME_DIR = "landmark_vietnam_dataset_split"

In [ ]:
# Đường dẫn gốc đến dataset (theo đường dẫn chuẩn của Kaggle)
# Thường dataset được mount tại /kaggle/input/<tên-dataset>
# Ở đây tên dataset là "landmark-mqa-dataset"
base_path = BASE_DATA

print("Kiểm tra đường dẫn:", base_path)
print("Có tồn tại?", os.path.exists(base_path))
print("Các thư mục con:")
if os.path.exists(base_path):
    for item in os.listdir(base_path):
        print(" -", item)
else:
    # Nếu không tìm thấy, thử đường dẫn có thêm "datasets"
    alt_path = BASE_DATA
    print(f"Thử đường dẫn thay thế: {alt_path}")
    if os.path.exists(alt_path):
        base_path = alt_path
        print("Đã tìm thấy tại:", base_path)
        for item in os.listdir(base_path):
            print(" -", item)
    else:
        print("Không tìm thấy dataset. Vui lòng kiểm tra lại tên dataset.")

In [ ]:
# Sau khi xác định base_path từ cell trên, chúng ta đặt lại các biến
# Giả sử base_path đã được gán đúng (từ cell trên)
# Nếu base_path chưa được định nghĩa, gán lại bằng tay
if 'base_path' not in dir():
    base_path = BASE_INPUT
    # Nếu không có, thử đường dẫn khác
    if not os.path.exists(base_path):
        base_path = BASE_DATA

# Tìm file json
json_file = None
for f in os.listdir(base_path):
    if f.endswith('.jsonl'):
        json_file = os.path.join(base_path, f)
        break
if json_file is None:
    # Có thể file nằm trong thư mục con
    for root, dirs, files in os.walk(base_path):
        for f in files:
            if f.endswith('.jsonl'):
                json_file = os.path.join(root, f)
                break
        if json_file:
            break

if json_file is None:
    raise FileNotFoundError("Không tìm thấy file train.jsonl trong dataset")

print("Đã tìm thấy file json:", json_file)

# Tìm thư mục ảnh (thường là thư mục con chứa các folder số)
image_dir = None
for item in os.listdir(base_path):
    if os.path.isdir(os.path.join(base_path, item)):
        # Kiểm tra xem có folder số bên trong không
        subdir = os.path.join(base_path, item)
        if any(f.isdigit() for f in os.listdir(subdir)):
            image_dir = subdir
            break
if image_dir is None:
    # Có thể thư mục ảnh là "landmark_dataset"
    if os.path.exists(os.path.join(base_path, NAME_DIR)):
        image_dir = os.path.join(base_path, NAME_DIR)
    else:
        image_dir = base_path  # fallback

print("Thư mục ảnh:", image_dir)

In [ ]:
def load_jsonl_to_df(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

def prepare_hf_dataset(df, split_name):
    ds = Dataset.from_pandas(df)
    
    # Hàm fix path: Kết hợp IMAGE_BASE_DIR + split (train/val/test) + image_path trong JSON
    def fix_path(example):
        example["image_path"] = os.path.join(image_dir, split_name, example["image"])
        return example

    ds = ds.map(fix_path)
    # Cast cột image_path thành kiểu Image của HuggingFace để model Qwen đọc được
    ds = ds.cast_column("image_path", Image())
    return ds

# --- THỰC THI LOAD 3 TẬP ---

# 1. Load Train (Đã bao gồm ảnh Augment vì lấy từ folder train)
df_train = load_jsonl_to_df(os.path.join(BASE_DATA, "train_vietnam_landmark.jsonl"))
train_dataset = prepare_hf_dataset(df_train, "train")

# 2. Load Val (Tập sạch)
df_val = load_jsonl_to_df(os.path.join(BASE_DATA, "val_vietnam_landmark.jsonl"))
val_dataset = prepare_hf_dataset(df_val, "val")

# 3. Load Test (Tập sạch)
df_test = load_jsonl_to_df(os.path.join(BASE_DATA, "test_vietnam_landmark.jsonl"))
test_dataset = prepare_hf_dataset(df_test, "test")

print(f"Train: {len(train_dataset)}")
print(f"Val: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")

# Prepare Model

In [ ]:
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

# Quantization 4-bit để tiết kiệm VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa"
)

# Chuẩn bị cho QLoRA
model = prepare_model_for_kbit_training(model)

In [ ]:
# Turn off vision parám
for name, param in model.named_parameters():
    if "vision" in name:
        param.requires_grad = False

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

processor = AutoProcessor.from_pretrained(model_id)

Load checkpoint to continue finetune

In [ ]:
# checkpoint_path = "/kaggle/input/datasets/kimngntrn510/qwen-checkpoint-900-vietnam"
# from peft import PeftModel
# model = PeftModel.from_pretrained(
#     model, 
#     checkpoint_path, 
#     is_trainable=True
# )

# processor = AutoProcessor.from_pretrained(checkpoint_path)

### Helper function

In [ ]:
import torch
from PIL import Image
import math

PATCH_SIZE = 28
TARGET_MAX_SIZE = 504  # 28 * 18 → ~324 tokens (ổn định, nhẹ VRAM)

def pad_to_square(img):
    w, h = img.size
    size = max(w, h)
    new_img = Image.new("RGB", (size, size), (0,0,0))
    new_img.paste(img, ((size - w)//2, (size - h)//2))
    return new_img

def smart_resize_by_tokens(img, target_tokens=196):
    w, h = img.size

    # số patch theo 1 chiều (sqrt token)
    target_grid = int(math.sqrt(target_tokens))

    # scale theo cạnh dài
    scale = (target_grid * PATCH_SIZE) / max(w, h)
    scale = min(scale, 1.0)  # ❗ không upscale

    new_w = int(w * scale)
    new_h = int(h * scale)

    # snap về bội số 28
    new_w = max(PATCH_SIZE, (new_w // PATCH_SIZE) * PATCH_SIZE)
    new_h = max(PATCH_SIZE, (new_h // PATCH_SIZE) * PATCH_SIZE)

    return img.resize((new_w, new_h), Image.Resampling.LANCZOS)


def collate_fn(examples):
    processed_messages = []

    for ex in examples:
        pil_img = ex["image_path"]

        pil_img = pad_to_square(pil_img)
        pil_img = smart_resize_by_tokens(pil_img)

        messages = ex["messages"]
        new_messages = []

        for msg in messages:
            new_content = []
            for c in msg["content"]:
                if c["type"] == "image":
                    new_content.append({
                        "type": "image",
                        "image": pil_img,
                    })
                else:
                    new_content.append(c)

            new_messages.append({
                "role": msg["role"],
                "content": new_content
            })

        processed_messages.append(new_messages)

    # text
    texts = [
        processor.apply_chat_template(
            msg,
            tokenize=False,
            add_generation_prompt=False
        )
        for msg in processed_messages
    ]

    # vision
    # image_inputs, video_inputs = process_vision_info(processed_messages)

    image_inputs = []
    for msgs in processed_messages:
        for m in msgs:
            for c in m["content"]:
                if c["type"] == "image":
                    image_inputs.append(c["image"])

    video_inputs = None

    inputs = processor(
        text=texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
        max_pixels=392 * 392,
        min_pixels = 392 * 392,
    )

    processor.image_processor.do_resize = False

    labels = inputs["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    inputs["labels"] = labels

    # chống NaN
    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)

    return inputs

In [ ]:
from transformers import TrainerCallback

class NaNCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            loss = logs.get("loss")
            if loss is not None and (loss != loss or loss > 100): # Check NaN hoặc Loss vọt quá cao
                print(f"\n[CẢNH BÁO] Phát hiện Loss bất thường: {loss} tại Step {state.global_step}")
                control.should_training_stop = True

# Training Model

In [ ]:
training_args = SFTConfig(
    # output_dir = "./debug_run",
    output_dir="./qwen2_5_vl_landmark",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    warmup_steps=30,
    optim="paged_adamw_8bit",
    fp16=False,
    bf16=True,
    
    num_train_epochs=2,
    logging_steps=50,
    eval_strategy="steps",          # thay evaluation_strategy bằng eval_strategy
    eval_steps=50,
    save_steps=100,
    
    load_best_model_at_end=True,
    remove_unused_columns=False,
    gradient_checkpointing=True,
    dataloader_pin_memory=False,

    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    data_collator=collate_fn,
    callbacks=[NaNCallback(),
               EarlyStoppingCallback(
                   early_stopping_patience=5,
                   early_stopping_threshold=0.0001
                )
              ]
)

In [ ]:
# FIRST TRAIN
#trainer.train()

In [ ]:
# FOR CONTINUE TRAINING
trainer.train(resume_from_checkpoint=checkpoint_path)

Save last checkpoint for continue finetune

In [ ]:
import shutil
checkpoint_path = '/kaggle/working/qwen2_5_vl_landmark/checkpoint-1000'

output_filename = '/kaggle/working/qwen_checkpoint_1000_vietnam'


if os.path.exists(checkpoint_path):
    shutil.make_archive(output_filename, 'zip', checkpoint_path)
    print("Done")


In [ ]:
# Sau khi train xong, model tốt nhất đã được load nhờ load_best_model_at_end
best_model_path = "./qwen2_5_vl_landmark_best"
trainer.save_model(best_model_path)
processor.save_pretrained(best_model_path)

In [ ]:
shutil.make_archive(best_model_path, 'zip', best_model_path)